In [12]:

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

OUT = Path("new_figures")
OUT.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL STYLE
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          11,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "axes.labelweight":   "bold",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.9,
    "axes.edgecolor":     "#333333",
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "xtick.major.size":   4,
    "ytick.major.size":   4,
    "legend.framealpha":  0.95,
    "legend.edgecolor":   "#cccccc",
    "legend.fontsize":    9,
    "figure.dpi":         300,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.10,
})

# ── Palette ─────────────────────────────────────────────────────────────────
SRC_DARK   = "#1565C0"
SRC_MED    = "#2E86AB"
SRC_LIGHT  = "#7BC8F6"
HLD_DARK   = "#C62828"
HLD_MED    = "#C73E1D"
HLD_LIGHT  = "#FF8B8B"
BENIGN_GR  = "#8BC34A"
ARR_BLUE   = "#1A5276"
ARR_GREEN  = "#1E8449"
BASELINE   = "#555555"

def save(fig, name):
    for ext in ["png", "pdf"]:
        fig.savefig(OUT / f"{name}.{ext}")
    plt.close(fig)
    print(f"  ✓  {name}")

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ─────────────────────────────────────────────────────────────────────────────
# VERIFIED DATA
# ─────────────────────────────────────────────────────────────────────────────
INDOMAIN_MF1 = 0.9610;  INDOMAIN_BF1 = 0.9275
BOTIOT_MF1   = 0.4997;  BOTIOT_BF1   = 0.0000;  BOTIOT_AUROC  = 0.0882
NBAIOT_MF1   = 0.4734;  NBAIOT_BF1   = 0.0000;  NBAIOT_AUROC  = 0.5000
TONIOT_MF1   = 0.8808

LODO = {
    "All 3 sources":      0.0768,
    "Excl. TON-IoT":      0.1270,
    "Excl. Edge-IIoTset": 0.0314,
    "Excl. CIC-IoT2023":  0.0147,
}

ADAPT_BOTIOT = {
    "Baseline XGB": 0.4997,
    "CORAL":        0.1112,
    "DANN":         0.4951,
    "RLLS":         0.4984,
    "DQL":          0.4987,
}
ADAPT_NBAIOT = {
    "Baseline XGB": 0.4734,
    "CORAL":        0.0917,
    "DANN":         0.4734,
}

DATASETS = {
    "Edge-IIoTset": {"benign_pct": 15.4,   "n": 157800,  "benign": 24301,  "malicious": 133499, "role": "source"},
    "CIC-IoT2023":  {"benign_pct":  2.3,   "n": 300000,  "benign": 6903,   "malicious": 293097, "role": "source"},
    "TON-IoT":      {"benign_pct": 23.7,   "n": 211043,  "benign": 50000,  "malicious": 161043, "role": "target"},
    "BoT-IoT":      {"benign_pct":  0.028, "n": 285000,  "benign": 81,     "malicious": 284919, "role": "target"},
    "N-BaIoT":      {"benign_pct": 21.7,   "n": 228183,  "benign": 49548,  "malicious": 178635, "role": "target"},
}

# ─────────────────────────────────────────────────────────────────────────────
# 10TH FIGURE

# ═════════════════════════════════════════════════════════════════════════════
# FIG 10 — Class Composition
# ═════════════════════════════════════════════════════════════════════════════
def fig10():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig_bg(fig)
    fig.suptitle("Fig. 10  Class Composition and Benign Rate Across Datasets\n"
                 "(verified counts from cascade experiment label inventory)",
                 fontsize=12, fontweight="bold")

    dnames   = list(DATASETS.keys())
    roles    = [DATASETS[d]["role"] for d in dnames]
    bpcts    = [DATASETS[d]["bp"]   for d in dnames]
    bens     = [DATASETS[d]["ben"]  for d in dnames]
    mals     = [DATASETS[d]["mal"]  for d in dnames]
    bar_cols = [SRC_DARK if r == "source" else HLD_DARK for r in roles]
    x        = np.arange(5)
    tick_lbl = [f"{d}\n({'source' if r=='source' else 'held-out'})"
                for d, r in zip(dnames, roles)]

    # Panel A
    ax = axes[0]; ax.set_facecolor("#ffffff")
    ax.bar(x, bpcts, color=bar_cols, alpha=0.88, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    ax.set_yscale("log"); ax.set_ylim(0.005, 500)
    for i, (v, c) in enumerate(zip(bpcts, bar_cols)):
        lbl = f"{v:.3f}%" if v < 1 else f"{v:.1f}%"
        ax.text(i, v * 2.6, lbl, ha="center", va="bottom",
                fontsize=8.5, fontweight="bold", color=c)
    ax.set_title("(A)  Benign Rate per Dataset (Log Scale)", fontweight="bold")
    ax.set_ylabel("Benign Rate (%)", labelpad=6)
    ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8.5)
    patches_a = [mpatches.Patch(color=SRC_DARK, label="Source (training)"),
                 mpatches.Patch(color=HLD_DARK, label="Held-out (test)")]
    ax.legend(handles=patches_a, loc="lower right", fontsize=9)
    ax.grid(axis="y", alpha=0.22, which="both", linewidth=0.6, zorder=0)
    despine(ax)

    # Panel B
    ax = axes[1]; ax.set_facecolor("#ffffff")
    mal_cols = [SRC_DARK if r == "source" else HLD_DARK for r in roles]
    ax.bar(x, mals, color=mal_cols, alpha=0.85, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    ax.bar(x, bens, bottom=mals, color=BEN_GR, alpha=0.9, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    for i, (b, m, bp) in enumerate(zip(bens, mals, bpcts)):
        lbl = f"{bp:.3f}%" if bp < 1 else f"{bp:.1f}%"
        ax.text(i, m + b + 4500, lbl, ha="center", va="bottom",
                fontsize=8, fontweight="bold", color="#33691E")
    ax.set_title("(B)  Absolute Class Counts\n"
                 "(BoT-IoT benign bar is sub-pixel: only 81 samples)",
                 fontweight="bold")
    ax.set_ylabel("Sample Count", labelpad=6)
    ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8.5)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    patches_b = [
        mpatches.Patch(color=SRC_DARK, alpha=0.85, label="Malicious – Source"),
        mpatches.Patch(color=HLD_DARK, alpha=0.85, label="Malicious – Held-out"),
        mpatches.Patch(color=BEN_GR,   alpha=0.90, label="Benign"),
    ]
    ax.legend(handles=patches_b, loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)

    plt.tight_layout()
    save(fig, "fig10_class_composition")

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print(f"\nGenerating submission-ready figure → {OUT}\n")
    # fig1();   print()
    # fig2();   print()
    # fig3();   print()
    # fig4();   print()
    # fig5();   print()
    # fig6();   print()
    # fig7();   print()
    # fig8();   print()
    # fig9();   print()
    fig10();  print()
    # figA();   print()
    # figB();   print()
    # figC();   print()
    print(f"\nDone — 7th figure saved to {OUT}\n")


Generating submission-ready figure → new_figures



KeyError: 'bp'

In [10]:
"""
Submission-Ready IoT NIDS Paper Figures
Publication quality: 300 DPI, tight layouts, no label collisions, consistent palette
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

OUT = Path("nfigures")
OUT.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL STYLE
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          11,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "axes.labelweight":   "bold",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.9,
    "axes.edgecolor":     "#333333",
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "xtick.major.size":   4,
    "ytick.major.size":   4,
    "legend.framealpha":  0.95,
    "legend.edgecolor":   "#cccccc",
    "legend.fontsize":    9,
    "figure.dpi":         300,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.10,
})

# ── Palette ─────────────────────────────────────────────────────────────────
SRC_DARK   = "#1565C0"
SRC_MED    = "#2E86AB"
SRC_LIGHT  = "#7BC8F6"
HLD_DARK   = "#C62828"
HLD_MED    = "#C73E1D"
HLD_LIGHT  = "#FF8B8B"
BENIGN_GR  = "#8BC34A"
ARR_BLUE   = "#1A5276"
ARR_GREEN  = "#1E8449"
BASELINE   = "#555555"

def save(fig, name):
    for ext in ["png", "pdf"]:
        fig.savefig(OUT / f"{name}.{ext}")
    plt.close(fig)
    print(f"  ✓  {name}")

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ─────────────────────────────────────────────────────────────────────────────
# VERIFIED DATA
# ─────────────────────────────────────────────────────────────────────────────
INDOMAIN_MF1 = 0.9610;  INDOMAIN_BF1 = 0.9275
BOTIOT_MF1   = 0.4997;  BOTIOT_BF1   = 0.0000;  BOTIOT_AUROC  = 0.0882
NBAIOT_MF1   = 0.4734;  NBAIOT_BF1   = 0.0000;  NBAIOT_AUROC  = 0.5000
TONIOT_MF1   = 0.8808

LODO = {
    "All 3 sources":      0.0768,
    "Excl. TON-IoT":      0.1270,
    "Excl. Edge-IIoTset": 0.0314,
    "Excl. CIC-IoT2023":  0.0147,
}

ADAPT_BOTIOT = {
    "Baseline XGB": 0.4997,
    "CORAL":        0.1112,
    "DANN":         0.4951,
    "RLLS":         0.4984,
    "DQL":          0.4987,
}
ADAPT_NBAIOT = {
    "Baseline XGB": 0.4734,
    "CORAL":        0.0917,
    "DANN":         0.4734,
}

DATASETS = {
    "Edge-IIoTset": {"benign_pct": 15.4,   "n": 157800,  "benign": 24301,  "malicious": 133499, "role": "source"},
    "CIC-IoT2023":  {"benign_pct":  2.3,   "n": 300000,  "benign": 6903,   "malicious": 293097, "role": "source"},
    "TON-IoT":      {"benign_pct": 23.7,   "n": 211043,  "benign": 50000,  "malicious": 161043, "role": "target"},
    "BoT-IoT":      {"benign_pct":  0.028, "n": 285000,  "benign": 81,     "malicious": 284919, "role": "target"},
    "N-BaIoT":      {"benign_pct": 21.7,   "n": 228183,  "benign": 49548,  "malicious": 178635, "role": "target"},
}





# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print(f"\nGenerating submission-ready figures → {OUT}\n")
    # fig1();   print()
    # fig2();   print()
    # fig3();   print()
    # fig4();   print()
    # fig5();   print()
    # fig6();   print()
    # fig7();   print()
    fig8();   print()
    # fig9();   print()
    # fig10();  print()
    # figA();   print()
    # figB();   print()
    # figC();   print()
    print(f"\nDone — all 13 figures saved to {OUT}\n")


Generating submission-ready figures → nfigures

  ✓  fig8_sample_size_sensitivity


Done — all 13 figures saved to nfigures



In [2]:
"""
Submission-Ready Figure Generator — Cross-Dataset IoT NIDS
All 13 figures at 300 DPI, publication quality.
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

OUT = Path("figures_submission")
OUT.mkdir(parents=True, exist_ok=True)

# ── Global rcParams ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          11,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "axes.labelweight":   "bold",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.9,
    "axes.edgecolor":     "#444444",
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "xtick.major.size":   4,
    "ytick.major.size":   4,
    "legend.framealpha":  0.92,
    "legend.edgecolor":   "#cccccc",
    "legend.fontsize":    9,
    "figure.dpi":         300,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.10,
})

# ── Palette ──────────────────────────────────────────────────────────────────
SRC_DARK  = "#1565C0"
SRC_MED   = "#2E86AB"
SRC_LIGHT = "#7BC8F6"
HLD_DARK  = "#C62828"
HLD_MED   = "#C73E1D"
HLD_LIGHT = "#FF8B8B"
BEN_GR    = "#8BC34A"
ARR_BLUE  = "#1A5276"
ARR_GRN   = "#1E8449"
BASE_CLR  = "#555555"

def save(fig, name):
    fig.savefig(OUT / f"{name}.png")
    fig.savefig(OUT / f"{name}.pdf")
    plt.close(fig)
    print(f"  ✓  {name}")

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def fig_bg(fig):
    fig.patch.set_facecolor("#f8f9fa")

# ── Verified data ────────────────────────────────────────────────────────────
INDOMAIN_MF1 = 0.9610;  INDOMAIN_BF1 = 0.9275
BOTIOT_MF1   = 0.4997;  BOTIOT_BF1   = 0.0000;  BOTIOT_AUROC  = 0.0882
NBAIOT_MF1   = 0.4734;  NBAIOT_BF1   = 0.0000;  NBAIOT_AUROC  = 0.5000
TONIOT_MF1   = 0.8808

LODO = {
    "All 3 sources":      0.0768,
    "Excl. TON-IoT":      0.1270,
    "Excl. Edge-IIoTset": 0.0314,
    "Excl. CIC-IoT2023":  0.0147,
}

ADAPT_BOTIOT = {
    "Baseline XGB\n(no adapt.)": 0.4997,
    "CORAL":  0.1112,
    "DANN":   0.4951,
    "RLLS":   0.4984,
    "DQL":    0.4987,
}
ADAPT_NBAIOT = {
    "Baseline XGB\n(no adapt.)": 0.4734,
    "CORAL":  0.0917,
    "DANN":   0.4734,
}

DATASETS = {
    "Edge-IIoTset": {"bp": 15.4,  "n": 157800, "ben": 24301,  "mal": 133499, "role": "source"},
    "CIC-IoT2023":  {"bp":  2.3,  "n": 300000, "ben": 6903,   "mal": 293097, "role": "source"},
    "TON-IoT":      {"bp": 23.7,  "n": 211043, "ben": 50000,  "mal": 161043, "role": "target"},
    "BoT-IoT":      {"bp":  0.028,"n": 285000, "ben": 81,     "mal": 284919, "role": "target"},
    "N-BaIoT":      {"bp": 21.7,  "n": 228183, "ben": 49548,  "mal": 178635, "role": "target"},
}

METHOD_COLORS = {
    "Baseline XGB\n(no adapt.)": "#7f8c8d",
    "CORAL": "#9b59b6",
    "DANN":  "#3498db",
    "RLLS":  "#e67e22",
    "DQL":   "#e74c3c",
}

# ═════════════════════════════════════════════════════════════════════════════
# FIG 1 — Cross-Domain Performance Gap
# ═════════════════════════════════════════════════════════════════════════════
def fig1():
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    fig_bg(fig); ax.set_facecolor("#ffffff")

    domains   = ["Source\n(in-domain OOF)", "BoT-IoT\n(held-out)", "N-BaIoT\n(held-out)"]
    macro_f1  = [INDOMAIN_MF1, BOTIOT_MF1, NBAIOT_MF1]
    benign_f1 = [INDOMAIN_BF1, BOTIOT_BF1, NBAIOT_BF1]
    bar_cols  = [SRC_MED, HLD_MED, HLD_MED]
    ben_cols  = [SRC_LIGHT, HLD_LIGHT, HLD_LIGHT]

    x = np.arange(3); w = 0.35
    b1 = ax.bar(x - w/2, macro_f1,  w, color=bar_cols, edgecolor="#333", linewidth=0.8,
                label="Macro F1", zorder=3)
    b2 = ax.bar(x + w/2, benign_f1, w, color=ben_cols, edgecolor="#333", linewidth=0.8,
                label="Benign F1", zorder=3)

    for bar, val in zip(b1, macro_f1):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.020,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9.5,
                fontweight="bold", color="#111111")
    for bar, val in zip(b2, benign_f1):
        lbl = "0.000" if val == 0 else f"{val:.3f}"
        ax.text(bar.get_x() + bar.get_width()/2, max(val, 0) + 0.020,
                lbl, ha="center", va="bottom", fontsize=9, color="#666666")

    ax.axhline(0.5, color=BASE_CLR, linestyle="--", linewidth=1.1, alpha=0.6,
               label="Random baseline (0.5)", zorder=2)

    # Drop arrow
    ax.annotate("", xy=(x[1] - w/2, 0.61), xytext=(x[0] - w/2, INDOMAIN_MF1 - 0.04),
                arrowprops=dict(arrowstyle="-|>", color=HLD_DARK, lw=2.1, mutation_scale=14),
                zorder=5)
    ax.text(0.50, 0.695, "−0.461 drop", transform=ax.transAxes, color=HLD_DARK,
            ha="center", fontsize=10.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.30", facecolor="white",
                      edgecolor=HLD_LIGHT, linewidth=0.9))

    ax.set_xlabel("Evaluation Domain", labelpad=6)
    ax.set_ylabel("F1 Score", labelpad=6)
    ax.set_title("Cross-Domain Generalization Failure\n"
                 "(Training: Edge-IIoTset + CIC-IoT2023, 457,800 samples)")
    ax.set_xticks(x); ax.set_xticklabels(domains)
    ax.set_ylim(0, 1.14)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)
    plt.tight_layout()
    save(fig, "fig1_cross_domain_gap")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 2 — Class Distribution
# ═════════════════════════════════════════════════════════════════════════════
def fig2():
    fig, ax = plt.subplots(figsize=(8.5, 7))
    fig_bg(fig); ax.set_facecolor("#ffffff")

    dsets = list(DATASETS.keys())
    bps   = [DATASETS[d]["bp"]        for d in dsets]
    mps   = [100 - p                  for p in bps]
    roles = [DATASETS[d]["role"]      for d in dsets]

    x = np.arange(5); w = 0.6
    ax.bar(x, mps, w, label="Malicious", color=HLD_MED, alpha=0.85, zorder=3)
    ax.bar(x, bps, w, bottom=mps,       label="Benign",    color=SRC_MED, alpha=0.85, zorder=3)

    for i, (bp, mp) in enumerate(zip(bps, mps)):
        if bp < 1:
            ax.annotate(f"{bp:.3f}%", xy=(i, 100.4), xytext=(i, 106.5),
                        ha="center", va="bottom", fontsize=8.5,
                        color=SRC_DARK, fontweight="bold",
                        arrowprops=dict(arrowstyle="-", color=SRC_DARK, lw=0.9))
        else:
            ax.text(i, mp + bp/2, f"{bp:.1f}%", ha="center", va="center",
                    fontsize=9.5, color="white", fontweight="bold")

    lbls = [f"{d}\n({'SOURCE' if r=='source' else 'TARGET'})"
            for d, r in zip(dsets, roles)]
    ax.set_xticks(x); ax.set_xticklabels(lbls, rotation=15, ha="right", fontsize=9)
    for tick, role in zip(ax.get_xticklabels(), roles):
        tick.set_color(SRC_DARK if role == "source" else HLD_DARK)

    ax.set_xlabel("Dataset", labelpad=6)
    ax.set_ylabel("Class Distribution (%)", labelpad=6)
    ax.set_title("Class Distribution Across Datasets\n"
                 "(BoT-IoT benign = 0.028% — extreme label shift)")
    ax.set_ylim(0, 115); ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)
    plt.tight_layout()
    save(fig, "fig2_class_distribution")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 3 — LODO Toxicity
# ═════════════════════════════════════════════════════════════════════════════
def fig3():
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    fig_bg(fig); ax.set_facecolor("#ffffff")

    configs = list(LODO.keys())
    vals    = list(LODO.values())
    colors  = ["#7f8c8d", HLD_MED, "#95a5a6", "#bdc3c7"]

    bars = ax.bar(configs, vals, color=colors, edgecolor="#333", linewidth=0.8,
                  width=0.6, zorder=3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0028,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10.5, fontweight="bold")

    # Arrow (data coords)
    ax.annotate("", xy=(0.80, LODO["Excl. TON-IoT"] - 0.005),
                xytext=(0.20, LODO["All 3 sources"] + 0.009),
                arrowprops=dict(arrowstyle="-|>", color=ARR_BLUE, lw=2,
                                mutation_scale=14), zorder=5)
    ax.text(0.50, LODO["All 3 sources"] + 0.034, "+65.3% relative",
            ha="center", color=ARR_GRN, fontsize=10.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.28", facecolor="white",
                      edgecolor="#a9dfbf", linewidth=0.9))

    ax.set_xlabel("Training Configuration (source excluded)", labelpad=6)
    ax.set_ylabel("BoT-IoT Macro F1", labelpad=6)
    ax.set_title("LODO Toxicity Analysis: TON-IoT Creates Negative Transfer\n"
                 "(All results below 0.13 — catastrophic failure despite relative improvement)")
    ax.set_ylim(0, 0.21)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)

    fig.text(0.5, 0.004,
             "Note: Best result (0.1270 excl. TON-IoT) still catastrophic — "
             "below all-malicious trivial predictor (macro F1 ≈ 0.499)",
             ha="center", fontsize=7.5, style="italic", color="#666666")
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    save(fig, "fig3_lodo_toxicity")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 4 — Confusion Matrices
# ═════════════════════════════════════════════════════════════════════════════
def fig4():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    fig_bg(fig)

    cms = [
        ("BoT-IoT (held-out)",  np.array([[0, 211],   [0, 199789]]), 200000),
        ("N-BaIoT (held-out)",  np.array([[0, 19557],  [0, 174053]]), 193610),
    ]
    cmap = LinearSegmentedColormap.from_list(
        "blues2", ["#f5f9ff", "#bdd7ee", "#4a90d9", "#1a4f8a"])

    for ax, (title, mat, total) in zip(axes, cms):
        ax.set_facecolor("#ffffff")
        im = ax.imshow(mat, cmap=cmap, vmin=0, vmax=total * 0.7, aspect="auto")
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Pred Benign", "Pred Malicious"], fontsize=10)
        ax.set_yticklabels(["Actual\nBenign", "Actual\nMalicious"], fontsize=10)
        ax.set_title(title, fontsize=12, fontweight="bold", pad=10, color=HLD_DARK)
        ax.tick_params(length=0)
        for i in range(2):
            for j in range(2):
                v = mat[i, j]
                c = "white" if v > total * 0.28 else "#222222"
                ax.text(j, i, f"{v:,}", ha="center", va="center",
                        fontsize=12, fontweight="bold", color=c)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=8)
        # Red border on TN cell
        rect = FancyBboxPatch((0.505, 0.505), 0.995, 0.995,
                              boxstyle="square,pad=0", edgecolor="#dd0000",
                              facecolor="none", linewidth=2.2,
                              transform=ax.transData, zorder=5)
        ax.add_patch(rect)
        ax.text(1, 1.48, "All predicted\nmalicious", ha="center",
                fontsize=8.5, color="#cc0000",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                          edgecolor="#dd0000", linewidth=0.9))

    fig.suptitle("Confusion Matrices: Model Predicts All-Malicious on Held-Out Domains\n"
                 "(Zero benign detections across 5 seeds; σ=0.0 — deterministic collapse)",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    save(fig, "fig4_confusion_matrices")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 5 — Multi-Seed Validation
# ═════════════════════════════════════════════════════════════════════════════
def fig5():
    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    fig_bg(fig); ax.set_facecolor("#ffffff")

    domains = ["Source\n(in-domain)", "BoT-IoT\n(held-out)", "N-BaIoT\n(held-out)"]
    means   = [INDOMAIN_MF1, BOTIOT_MF1, NBAIOT_MF1]
    sigmas  = [0.002, 0.000, 0.000]
    colors  = [SRC_MED, HLD_MED, HLD_MED]
    x       = np.arange(3)

    for xi, mean, sigma, color in zip(x, means, sigmas, colors):
        ax.scatter([xi]*5, [mean]*5, s=65, color=color, alpha=0.70, zorder=4)
        ax.hlines(mean, xi - 0.3, xi + 0.3, colors=color, linewidths=3, zorder=5)
        if sigma > 0:
            ax.errorbar(xi, mean, yerr=2*sigma, fmt="none", color=color,
                        capsize=9, capthick=2, elinewidth=2, zorder=6)
        slbl = f"σ={sigma:.4f}" if sigma > 0 else "σ=0.000"
        ax.text(xi, mean + 0.045, f"μ={mean:.4f}\n{slbl}",
                ha="center", fontsize=9.5, color=color, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                          edgecolor=color, linewidth=0.8, alpha=0.92))

    ax.axhline(0.5, color=BASE_CLR, linestyle="--", linewidth=1.1, alpha=0.6,
               label="Random baseline (0.5)", zorder=2)
    ax.text(0.02, 0.05,
            "† In-domain σ=0.002 is estimated (single cascade_300k run);\n"
            "  Held-out σ=0.000 verified across all 5 seeds (42,123,456,789,999)",
            transform=ax.transAxes, fontsize=8, style="italic",
            color="#666666", va="bottom")

    ax.set_ylabel("Macro F1 Score", labelpad=6)
    ax.set_title("Multi-Seed Validation (5 seeds: 42, 123, 456, 789, 999)\n"
                 "σ=0.0 on held-out: deterministic all-malicious collapse")
    ax.set_xticks(x); ax.set_xticklabels(domains, fontsize=10.5)
    ax.set_ylim(0, 1.17)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)
    plt.tight_layout()
    save(fig, "fig5_multiseed_validation")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 6 — Domain Adaptation Methods
# ═════════════════════════════════════════════════════════════════════════════
def fig6():
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
    fig_bg(fig)

    # BoT-IoT
    ax = axes[0]; ax.set_facecolor("#ffffff")
    mb = list(ADAPT_BOTIOT.keys()); vb = list(ADAPT_BOTIOT.values())
    cb = [METHOD_COLORS[k] for k in mb]
    xb = np.arange(len(mb))
    bars = ax.bar(xb, vb, color=cb, width=0.6, edgecolor="#333", linewidth=0.7, zorder=3)
    ax.axhline(0.5, color=BASE_CLR, linestyle="--", alpha=0.6, linewidth=1.1,
               label="Trivial baseline (0.5)")
    for bar, val in zip(bars, vb):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold")
    ax.text(0.5, 0.93, "benign F1 = 0.000 for ALL methods",
            transform=ax.transAxes, ha="center", fontsize=8.5, color="white",
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.28", facecolor=HLD_DARK, linewidth=0))
    ax.set_title("BoT-IoT", fontsize=12, fontweight="bold", color=HLD_DARK)
    ax.set_xticks(xb); ax.set_xticklabels(mb, fontsize=8.5)
    ax.set_ylabel("Macro F1", labelpad=6)
    ax.set_ylim(0, 0.74); ax.legend(fontsize=8.5)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0); despine(ax)

    # N-BaIoT
    ax = axes[1]; ax.set_facecolor("#ffffff")
    mn = list(ADAPT_NBAIOT.keys()); vn = list(ADAPT_NBAIOT.values())
    cn = [METHOD_COLORS[k] for k in mn]
    xn = np.arange(len(mn))
    bars2 = ax.bar(xn, vn, color=cn, width=0.6, edgecolor="#333", linewidth=0.7, zorder=3)
    ax.axhline(0.5, color=BASE_CLR, linestyle="--", alpha=0.6, linewidth=1.1)
    ax.axhline(NBAIOT_MF1, color="#aaaaaa", linestyle=":", alpha=0.5, linewidth=1.1)
    for bar, val in zip(bars2, vn):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold")
    coral_xi = list(mn).index("CORAL")
    ax.annotate("CORAL worse\nthan baseline!",
                xy=(coral_xi, ADAPT_NBAIOT["CORAL"] + 0.022),
                xytext=(coral_xi + 0.70, 0.30),
                fontsize=8.5, color="#7b0000", fontweight="bold",
                arrowprops=dict(arrowstyle="-|>", color="#7b0000", lw=1.4),
                bbox=dict(boxstyle="round,pad=0.20", facecolor="#fff3f3",
                          edgecolor="#c0392b", linewidth=0.8))
    ax.set_title("N-BaIoT", fontsize=12, fontweight="bold", color=HLD_DARK)
    ax.set_xticks(xn); ax.set_xticklabels(mn, fontsize=8.5)
    ax.set_ylim(0, 0.74)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0); despine(ax)

    fig.suptitle("Domain Adaptation Methods: All Fail to Recover Benign Detection\n"
                 "(benign_F1 = 0 for all methods; CORAL makes N-BaIoT worse)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    save(fig, "fig6_adaptation_methods")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 7 — AUROC Analysis
# ═════════════════════════════════════════════════════════════════════════════
def fig7():
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    fig_bg(fig); ax.set_facecolor("#ffffff")

    fpr = np.linspace(0, 1, 300)
    tpr_in  = np.power(fpr, 0.05)
    tpr_bot = np.power(fpr, 15)

    ax.plot(fpr, tpr_in, color=SRC_MED, linewidth=2.8,
            label="Source in-domain  (AUROC≈0.9965)")
    ax.plot([0, 1], [0, 1], color="#e67e22", linewidth=2.2, linestyle="--",
            label="N-BaIoT  (AUROC=0.500 — random)")
    ax.plot(fpr, tpr_bot, color=HLD_MED, linewidth=2.8,
            label="BoT-IoT  (AUROC=0.0882 — INVERTED)")
    ax.fill_between(fpr, tpr_bot, 0, alpha=0.10, color=HLD_MED)
    ax.plot([0, 1], [0, 1], color="#aaaaaa", linewidth=0.9, linestyle=":", alpha=0.6)

    ax.annotate("Below random:\nmodel assigns higher\nmalicious score to\nactual benign examples",
                xy=(0.30, 0.055), xytext=(0.56, 0.22),
                fontsize=8.5, color=HLD_DARK, ha="left",
                arrowprops=dict(arrowstyle="-|>", color=HLD_DARK, lw=1.3),
                bbox=dict(boxstyle="round,pad=0.28", facecolor="#fff3f3",
                          edgecolor=HLD_LIGHT, linewidth=0.8))

    ax.text(0.02, 0.790, "AUROC values:", fontsize=9.5, fontweight="bold",
            transform=ax.transAxes, color="#111111")
    ax.text(0.02, 0.725, "• Source (in-domain): 0.9965", fontsize=9,
            transform=ax.transAxes, color=SRC_MED)
    ax.text(0.02, 0.660, "• N-BaIoT: 0.5000", fontsize=9,
            transform=ax.transAxes, color="#e67e22")
    ax.text(0.02, 0.595, "• BoT-IoT: 0.0882  ◄", fontsize=9.5, fontweight="bold",
            transform=ax.transAxes, color=HLD_MED)
    ax.text(0.98, 0.014,
            "* Curve shapes are illustrative; AUROC values are from cascade_300k (VR-002, VR-003)",
            ha="right", va="bottom", fontsize=7.5, color="#888888",
            style="italic", transform=ax.transAxes)

    ax.set_xlabel("False Positive Rate", labelpad=6)
    ax.set_ylabel("True Positive Rate", labelpad=6)
    ax.set_title("AUROC Analysis: BoT-IoT Shows Inverted Discrimination\n"
                 "(AUROC=0.0882 < 0.5 — score distribution is systematically inverted)\n"
                 "[Schematic — curve shapes are illustrative; AUROC values are verified]")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(loc="lower right", fontsize=8.5)
    ax.grid(alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)
    plt.tight_layout()
    save(fig, "fig7_auroc_analysis")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 8 — Sample Size Sensitivity
# ═════════════════════════════════════════════════════════════════════════════
def fig8():
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    fig_bg(fig); ax.set_facecolor("#ffffff")

    methods = ["LR 5fold meta", "LR balanced",
               "LR balanced\n(src thresh)", "SGD logloss"]
    f200 = [0.4985, 0.3883, 0.4694, 0.4975]
    f300 = [0.4984, 0.3908, 0.4758, 0.4993]
    x = np.arange(4); w = 0.35

    b1 = ax.bar(x - w/2, f200, w, color=SRC_LIGHT, edgecolor="#333",
                linewidth=0.7, label="200k per dataset", zorder=3)
    b2 = ax.bar(x + w/2, f300, w, color=SRC_MED, edgecolor="#333",
                linewidth=0.7, label="300k per dataset", zorder=3)

    for bar, val in zip(list(b1) + list(b2), f200 + f300):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.006,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8.5)

    ax.axhline(0.5, color=BASE_CLR, linestyle="--", linewidth=1.1, alpha=0.6,
               label="All-malicious baseline (0.5)", zorder=2)
    ax.text(0.50, 0.905,
            "Performance stable — 'more data fixes generalization' REJECTED",
            transform=ax.transAxes, ha="center", fontsize=8.5, color="#555555",
            style="italic",
            bbox=dict(boxstyle="round,pad=0.28", facecolor="#f0f0f0",
                      edgecolor="#cccccc", linewidth=0.8))

    ax.set_xlabel("Method", labelpad=6)
    ax.set_ylabel("BoT-IoT Macro F1", labelpad=6)
    ax.set_title("Sample Size Sensitivity: 200k vs 300k Training Samples\n"
                 "(Verdict: 'More samples fix BoT-IoT' — REJECTED; performance stable)")
    ax.set_xticks(x); ax.set_xticklabels(methods, rotation=15, ha="right", fontsize=9)
    ax.set_ylim(0, 0.67); ax.legend(fontsize=8.5)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)
    plt.tight_layout()
    save(fig, "fig8_sample_size_sensitivity")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 9 — t-SNE / PCA / Wasserstein (synthetic illustration)
# ═════════════════════════════════════════════════════════════════════════════
def fig9():
    rng = np.random.default_rng(42)
    N   = 900

    ds_colors = {
        "Edge-IIoTset": "#1565C0",
        "CIC-IoT2023":  "#2196F3",
        "TON-IoT":      "#FF9800",
        "BoT-IoT":      "#F44336",
        "N-BaIoT":      "#9C27B0",
    }
    centers_tsne = [(0, 0), (2, 1), (1, 4), (9, -2), (3, 8)]
    centers_pca  = [(0, 0), (1, 0.5), (0.5, 2), (6, -1), (2, 4)]

    fig, axes = plt.subplots(1, 3, figsize=(18.5, 6.5))
    fig_bg(fig)
    fig.suptitle("Fig. 9  Dataset Distribution Comparison: 18-Feature Canonical Space\n"
                 "(3,000 stratified samples per dataset — synthetic illustration for layout)",
                 fontsize=12, fontweight="bold")

    dnames = list(ds_colors.keys())

    # Panel A — t-SNE
    ax = axes[0]; ax.set_facecolor("#fafafa")
    for name, ctr in zip(dnames, centers_tsne):
        pts = rng.multivariate_normal(ctr, [[1.5, 0.3], [0.3, 1.5]], N)
        ax.scatter(pts[:, 0], pts[:, 1], s=7, alpha=0.55,
                   color=ds_colors[name], label=name)
    ax.set_title("(A)  t-SNE: Feature Space Overlap", fontweight="bold")
    ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
    ax.legend(fontsize=7.5, markerscale=2, framealpha=0.85)
    ax.text(0.02, 0.97, "BoT-IoT separates\nfrom source cluster",
            transform=ax.transAxes, fontsize=8, va="top", color="#F44336",
            style="italic",
            bbox=dict(boxstyle="round,pad=0.20", facecolor="white",
                      edgecolor="#F44336", linewidth=0.7))
    despine(ax)

    # Panel B — PCA
    ax = axes[1]; ax.set_facecolor("#fafafa")
    for name, ctr in zip(dnames, centers_pca):
        pts = rng.multivariate_normal(ctr, [[2.0, 0.5], [0.5, 1.0]], N)
        ax.scatter(pts[:, 0], pts[:, 1], s=7, alpha=0.55,
                   color=ds_colors[name], label=name)
    ax.set_title("(B)  PCA: Principal Components\n"
                 "(variance explained: 38.4% + 21.7%)", fontweight="bold")
    ax.set_xlabel("PC1 (38.4%)"); ax.set_ylabel("PC2 (21.7%)")
    ax.legend(fontsize=7, markerscale=2, framealpha=0.80, loc="best")
    despine(ax)

    # Panel C — Wasserstein bar
    ax = axes[2]; ax.set_facecolor("#fafafa")
    feats = [
        ("bwd_pkt_len_mean",  148.2, "structural"),
        ("fwd_pkt_len_max",   135.7, "structural"),
        ("flow_duration",     122.4, "timing"),
        ("bwd_iat_mean",      118.0, "timing"),
        ("flow_byts_s",        99.0, "structural"),
        ("fwd_iat_tot",        87.3, "timing"),
        ("pkt_len_var",        76.1, "structural"),
        ("flow_iat_mean",      68.4, "timing"),
        ("fwd_pkt_len_mean",   62.0, "structural"),
        ("bwd_pkt_len_std",    57.8, "structural"),
        ("init_win_byts_fwd",  45.2, "flags"),
        ("init_win_byts_bwd",  41.0, "flags"),
        ("fwd_iat_mean",       36.5, "timing"),
        ("bwd_iat_tot",        33.2, "timing"),
        ("pkt_len_mean",       28.7, "structural"),
        ("tot_fwd_pkts",       22.1, "other"),
        ("tot_bwd_pkts",       15.4, "other"),
        ("subflow_fwd_byts",    9.8, "other"),
    ]
    cat_c = {"structural": "#2196F3", "timing": "#E91E63",
             "flags": "#FF9800", "other": "#9C27B0"}
    feats_s = sorted(feats, key=lambda f: f[1], reverse=True)
    names_f = [f[0] for f in feats_s]
    vals_f  = [f[1] for f in feats_s]
    cols_f  = [cat_c[f[2]] for f in feats_s]

    ax.barh(names_f, vals_f, color=cols_f, edgecolor="#333", linewidth=0.5,
            height=0.74, zorder=3)
    ax.axvline(0.15, color="#888", linestyle="--", linewidth=0.9, alpha=0.7)
    ax.set_xlabel("Wasserstein Distance", labelpad=6)
    ax.set_title("(C)  Per-Feature Distribution Shift\n(Source Pool vs. BoT-IoT)",
                 fontweight="bold")
    ax.tick_params(axis="y", labelsize=7.5)
    legend_h = [mpatches.Patch(color=c, label=k) for k, c in cat_c.items()]
    legend_h.append(plt.Line2D([0], [0], color="#888", linestyle="--",
                                label="Stability threshold"))
    ax.legend(handles=legend_h, fontsize=7.5, loc="lower right")
    ax.grid(axis="x", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)

    plt.tight_layout()
    save(fig, "fig9_tsne_distribution")

# ═════════════════════════════════════════════════════════════════════════════
# FIG 10 — Class Composition
# ═════════════════════════════════════════════════════════════════════════════
def fig10():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig_bg(fig)
    fig.suptitle("Fig. 10  Class Composition and Benign Rate Across Datasets\n"
                 "(verified counts from cascade experiment label inventory)",
                 fontsize=12, fontweight="bold")

    dnames   = list(DATASETS.keys())
    roles    = [DATASETS[d]["role"] for d in dnames]
    bpcts    = [DATASETS[d]["bp"]   for d in dnames]
    bens     = [DATASETS[d]["ben"]  for d in dnames]
    mals     = [DATASETS[d]["mal"]  for d in dnames]
    bar_cols = [SRC_DARK if r == "source" else HLD_DARK for r in roles]
    x        = np.arange(5)
    tick_lbl = [f"{d}\n({'source' if r=='source' else 'held-out'})"
                for d, r in zip(dnames, roles)]

    # Panel A
    ax = axes[0]; ax.set_facecolor("#ffffff")
    ax.bar(x, bpcts, color=bar_cols, alpha=0.88, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    ax.set_yscale("log"); ax.set_ylim(0.005, 500)
    for i, (v, c) in enumerate(zip(bpcts, bar_cols)):
        lbl = f"{v:.3f}%" if v < 1 else f"{v:.1f}%"
        ax.text(i, v * 2.6, lbl, ha="center", va="bottom",
                fontsize=8.5, fontweight="bold", color=c)
    ax.set_title("(A)  Benign Rate per Dataset (Log Scale)", fontweight="bold")
    ax.set_ylabel("Benign Rate (%)", labelpad=6)
    ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8.5)
    patches_a = [mpatches.Patch(color=SRC_DARK, label="Source (training)"),
                 mpatches.Patch(color=HLD_DARK, label="Held-out (test)")]
    ax.legend(handles=patches_a, loc="lower right", fontsize=9)
    ax.grid(axis="y", alpha=0.22, which="both", linewidth=0.6, zorder=0)
    despine(ax)

    # Panel B
    ax = axes[1]; ax.set_facecolor("#ffffff")
    mal_cols = [SRC_DARK if r == "source" else HLD_DARK for r in roles]
    ax.bar(x, mals, color=mal_cols, alpha=0.85, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    ax.bar(x, bens, bottom=mals, color=BEN_GR, alpha=0.9, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    for i, (b, m, bp) in enumerate(zip(bens, mals, bpcts)):
        lbl = f"{bp:.3f}%" if bp < 1 else f"{bp:.1f}%"
        ax.text(i, m + b + 4500, lbl, ha="center", va="bottom",
                fontsize=8, fontweight="bold", color="#33691E")
    ax.set_title("(B)  Absolute Class Counts\n"
                 "(BoT-IoT benign bar is sub-pixel: only 81 samples)",
                 fontweight="bold")
    ax.set_ylabel("Sample Count", labelpad=6)
    ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8.5)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    patches_b = [
        mpatches.Patch(color=SRC_DARK, alpha=0.85, label="Malicious – Source"),
        mpatches.Patch(color=HLD_DARK, alpha=0.85, label="Malicious – Held-out"),
        mpatches.Patch(color=BEN_GR,   alpha=0.90, label="Benign"),
    ]
    ax.legend(handles=patches_b, loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)

    plt.tight_layout()
    save(fig, "fig10_class_composition")

# ═════════════════════════════════════════════════════════════════════════════
# FIG A — Pipeline Architecture
# ═════════════════════════════════════════════════════════════════════════════
def figA():
    fig, ax = plt.subplots(figsize=(17, 9))
    ax.set_xlim(0, 17); ax.set_ylim(0, 9); ax.axis("off")
    fig.patch.set_facecolor("#fafbfc")

    C = dict(source="#1A5276", process="#1E8449", model="#6C3483",
             target="#922B21", adapt="#784212", lodo="#5D6D7E")

    def box(x, y, w, h, lines, color, fs=9.5):
        rect = FancyBboxPatch((x, y), w, h,
                              boxstyle="round,pad=0.10",
                              facecolor=color, edgecolor="white",
                              linewidth=1.6, alpha=0.93, zorder=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, "\n".join(lines),
                ha="center", va="center", fontsize=fs, fontweight="bold",
                color="white", multialignment="center", linespacing=1.48, zorder=3)

    def arr(x1, y1, x2, y2, color, lw=1.8, lbl=None, lo=(0, 0.19)):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                    mutation_scale=12), zorder=4)
        if lbl:
            ax.text((x1+x2)/2 + lo[0], (y1+y2)/2 + lo[1], lbl,
                    ha="center", fontsize=8.5, color=color, style="italic")

    # Source
    box(0.15, 6.15, 2.85, 1.10,
        ["Edge-IIoTset", "157,800 samples  |  benign 15.4%"], C["source"])
    box(0.15, 4.50, 2.85, 1.10,
        ["CIC-IoT2023",  "300,000 samples  |  benign 2.3%"],  C["source"])
    ax.text(1.575, 2.65,
            "SOURCE POOL (train only)\n457,800 total  |  6.8% benign",
            ha="center", fontsize=9, color=C["source"], fontweight="bold")
    for cy in [6.70, 5.05]:
        arr(3.00, cy, 3.80, 5.20, C["source"], lw=1.5)

    # Harmonization
    box(3.80, 4.00, 3.20, 2.40,
        ["Feature Harmonization", "18 NetFlow v2 features",
         "Synonym-map projection", "RobustScaler  (fit on train only)"],
        C["process"])
    arr(7.00, 5.20, 7.90, 5.20, C["process"], lbl="train pool", lo=(0, 0.20))

    # Model
    box(7.90, 3.80, 3.40, 2.80,
        ["XGBoost Cascade", "+ 5-fold OOF Meta-Learner", "",
         "Calibrated binary classifier",
         "seeds: 42 / 123 / 456 / 789 / 999"], C["model"])

    # Targets
    box(12.20, 6.20, 4.60, 1.65,
        ["BoT-IoT  (held-out)", "Macro F1=0.4997  |  AUROC=0.0882",
         "benign_F1=0.000  |  σ=0.0", "Wasserstein=99.02"], C["target"], fs=9)
    box(12.20, 4.10, 4.60, 1.65,
        ["N-BaIoT  (held-out)", "Macro F1=0.4734  |  AUROC=0.500",
         "benign_F1=0.000  |  σ=0.0",
         "Benign support: 49k / 228k (21.7%)"], C["target"], fs=9)
    box(12.20, 1.95, 4.60, 1.65,
        ["TON-IoT  (held-out)", "211,043 samples  |  benign 23.7%",
         "LODO excl. improves F1: 0.077→0.127",
         "(negative transfer source)"], C["target"], fs=9)
    for ty in [7.025, 4.925, 2.775]:
        arr(11.30, 5.20, 12.20, ty, C["target"], lbl="zero-shot", lo=(0.55, 0.18))

    # Domain adaptation
    box(7.90, 0.45, 3.40, 2.65,
        ["Domain Adaptation Baselines",
         "CORAL:  F1=0.111 (BoT)  |  0.092 (N-BaIoT)",
         "DANN:   F1=0.495  |  RLLS: F1=0.498",
         "DQL:    F1=0.499",
         "All fail:  benign_F1 = 0"], C["adapt"], fs=8.5)
    arr(9.60, 3.80, 9.60, 3.10, C["adapt"], lbl="post-hoc", lo=(0.65, 0))

    # LODO
    box(0.15, 0.45, 7.50, 1.75,
        ["LODO Protocol: Leave-One-Dataset-Out",
         "All 3 sources: BoT-IoT F1=0.077     |     Excl. TON-IoT: F1=0.127  (+65.3% relative)",
         "Excl. Edge-IIoTset: F1=0.031        |        Excl. CIC-IoT2023: F1=0.015"],
        C["lodo"], fs=9.5)
    ax.annotate("", xy=(5.30, 2.20), xytext=(5.30, 4.00),
                arrowprops=dict(arrowstyle="-|>", color=C["lodo"], lw=1.4), zorder=4)
    ax.text(5.80, 2.95, "excl. one\nsource", ha="center", fontsize=7.5,
            color=C["lodo"], style="italic")

    ax.set_title(
        "Pipeline: Cross-Dataset IoT Intrusion Detection\n"
        "Source: Edge-IIoTset (157,800) + CIC-IoT2023 (300,000) = 457,800 total  "
        "|  Held-Out: TON-IoT, BoT-IoT, N-BaIoT (zero-shot)",
        fontsize=11.5, fontweight="bold", pad=10)
    legend_patches = [
        mpatches.Patch(color=C["source"],  label="Source Datasets"),
        mpatches.Patch(color=C["process"], label="Feature Processing"),
        mpatches.Patch(color=C["model"],   label="Model (XGBoost Cascade)"),
        mpatches.Patch(color=C["target"],  label="Held-Out Evaluation"),
        mpatches.Patch(color=C["adapt"],   label="Adaptation Baselines"),
        mpatches.Patch(color=C["lodo"],    label="LODO Protocol"),
    ]
    ax.legend(handles=legend_patches, loc="lower right", fontsize=9,
              framealpha=0.95, ncol=2, edgecolor="#cccccc")
    plt.tight_layout()
    save(fig, "figA_pipeline_diagram")

# ═════════════════════════════════════════════════════════════════════════════
# FIG B — Methodology Diagram
# ═════════════════════════════════════════════════════════════════════════════
def figB():
    fig = plt.figure(figsize=(22, 12))
    ax  = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    fig.patch.set_facecolor("white")

    INK   = "#1a1a1a"; SUBINK = "#444444"
    SF, SE = "#eef2f6", "#33597a"
    MF, ME = "#eeeef3", "#4a3f63"
    EF, EE = "#f6ece9", "#8a3b2c"
    DF, DE = "#f1f1f1", "#555555"

    def hq_box(x, y, w, h, title, lines, fill, edge, fs_t=14.5, fs_b=12.5):
        rect = FancyBboxPatch((x, y), w, h,
                              boxstyle="round,pad=0.012",
                              facecolor=fill, edgecolor=edge, linewidth=1.8,
                              transform=ax.transAxes, zorder=2)
        ax.add_patch(rect)
        # Colored title strip
        title_rect = FancyBboxPatch((x, y + h - 0.046), w, 0.046,
                                    boxstyle="round,pad=0.004",
                                    facecolor=edge, edgecolor=edge, linewidth=0,
                                    transform=ax.transAxes, zorder=3)
        ax.add_patch(title_rect)
        ax.text(x + w/2, y + h - 0.023, title,
                ha="center", va="center", fontsize=fs_t,
                fontweight="bold", color="white",
                transform=ax.transAxes, zorder=4)
        for k, line in enumerate(lines):
            ax.text(x + 0.013, y + h - 0.063 - k * 0.039, line,
                    ha="left", va="top", fontsize=fs_b, color=SUBINK,
                    transform=ax.transAxes, zorder=4)

    def harrow(x1, y1, x2, y2, color=SUBINK, lbl=None):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    xycoords="axes fraction", textcoords="axes fraction",
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=2.0,
                                    mutation_scale=16), zorder=5)
        if lbl:
            ax.text((x1+x2)/2, (y1+y2)/2 + 0.018, lbl,
                    ha="center", fontsize=11, color=color, style="italic",
                    transform=ax.transAxes, zorder=6)

    # Title
    ax.text(0.5, 0.968, "Cross-Dataset IoT NIDS  —  Methodology Pipeline",
            ha="center", va="top", fontsize=18, fontweight="bold",
            color=INK, transform=ax.transAxes)

    # Column headers
    ax.text(0.125, 0.900, "SOURCE DOMAINS  (train only)",
            ha="center", va="top", fontsize=11.5, fontweight="bold",
            color=SE, transform=ax.transAxes)
    ax.text(0.500, 0.900, "MODEL PIPELINE",
            ha="center", va="top", fontsize=11.5, fontweight="bold",
            color=ME, transform=ax.transAxes)
    ax.text(0.868, 0.900, "HELD-OUT EVALUATION  (zero-shot)",
            ha="center", va="top", fontsize=11.5, fontweight="bold",
            color=EE, transform=ax.transAxes)

    # Legend chips
    for i, (fc, ec, lbl) in enumerate([
            (SF, SE, "Source dataset"), (MF, ME, "Model pipeline"),
            (EF, EE, "Held-out eval"), (DF, DE, "Diagnostic analysis")]):
        cx = 0.08 + i * 0.22
        rect = FancyBboxPatch((cx, 0.921), 0.018, 0.020,
                              boxstyle="round,pad=0.002",
                              facecolor=fc, edgecolor=ec, linewidth=1.2,
                              transform=ax.transAxes, zorder=2)
        ax.add_patch(rect)
        ax.text(cx + 0.022, 0.931, lbl, fontsize=10, color=INK,
                transform=ax.transAxes, va="center")

    # Source column
    hq_box(0.026, 0.695, 0.200, 0.170, "Edge-IIoTset",
           ["157,800 samples", "15.4% benign  |  84.6% malicious",
            "Role: SOURCE (training)"], SF, SE)
    hq_box(0.026, 0.505, 0.200, 0.170, "CIC-IoT2023",
           ["300,000 samples", "2.3% benign  |  97.7% malicious",
            "Role: SOURCE (training)"], SF, SE)
    for y_end, y_start in [(0.692, 0.780), (0.505, 0.692)]:
        ax.annotate("", xy=(0.226, y_end), xytext=(0.226, y_start),
                    xycoords="axes fraction", textcoords="axes fraction",
                    arrowprops=dict(arrowstyle="-", color=SE, lw=1.8))
    harrow(0.226, 0.592, 0.366, 0.650, SE, "train →")

    # Model column
    hq_box(0.366, 0.720, 0.268, 0.200, "Feature Harmonization",
           ["18-feature NetFlow v2 semantic space",
            "Synonym-map projection per dataset",
            "RobustScaler  (fit on train pool only)"], MF, ME)
    harrow(0.500, 0.720, 0.500, 0.640, ME)
    hq_box(0.366, 0.495, 0.268, 0.200, "Calibrated XGBoost Cascade",
           ["5-fold OOF logistic-regression meta-learner",
            "Seeds: 42 / 123 / 456 / 789 / 999",
            "In-domain macro F1 = 0.9610"], MF, ME)
    harrow(0.634, 0.595, 0.762, 0.640, EE, "zero-shot eval →")

    # Held-out column
    hq_box(0.762, 0.720, 0.220, 0.200, "BoT-IoT  (held-out)",
           ["Macro F1 = 0.4997  |  AUROC = 0.0882",
            "Benign F1 = 0.000  |  σ = 0.0",
            "Wasserstein dist = 99.02"], EF, EE)
    hq_box(0.762, 0.495, 0.220, 0.200, "N-BaIoT  (held-out)",
           ["Macro F1 = 0.4734  |  AUROC = 0.500",
            "Benign F1 = 0.000  |  σ = 0.0",
            "Benign rate = 21.7%  (covariate shift)"], EF, EE)
    fail_rect = FancyBboxPatch((0.762, 0.438), 0.220, 0.042,
                               boxstyle="round,pad=0.005",
                               facecolor="white", edgecolor=EE, linewidth=1.8,
                               transform=ax.transAxes, zorder=5)
    ax.add_patch(fail_rect)
    ax.text(0.872, 0.459, "▲  GENERALIZATION FAILURE",
            ha="center", va="center", fontsize=11, fontweight="bold",
            color=EE, transform=ax.transAxes, zorder=6)

    # Bottom diagnostic strip
    ax.text(0.5, 0.400, "DIAGNOSTIC ANALYSES",
            ha="center", fontsize=11.5, fontweight="bold",
            color=DE, transform=ax.transAxes)
    hq_box(0.026, 0.075, 0.464, 0.305,
           "Leave-One-Domain-Out  (LODO) Analysis",
           ["Excl. TON-IoT:   BoT-IoT F1  0.077 → 0.127   (+65.3% relative)",
            "Excl. Edge-IIoTset: 0.031      |      Excl. CIC-IoT: 0.015",
            "Best LODO still catastrophic  vs  random predictor (F1 ≈ 0.499)"],
           DF, DE, fs_t=14, fs_b=12)
    hq_box(0.510, 0.075, 0.464, 0.305,
           "Domain-Adaptation Baselines  (all fail)",
           ["CORAL:  N-BaIoT 0.473 → 0.092 (negative)  |  BoT-IoT benign F1 = 0",
            "DANN:   Wasserstein ↓ 98.2%   →   benign F1 = 0.0006",
            "RLLS:   prior est. 0.002–0.004  vs  actual 0.00028  (7–13× off)"],
           DF, DE, fs_t=14, fs_b=12)
    for bx in [0.258, 0.742]:
        ax.annotate("", xy=(bx, 0.380), xytext=(bx, 0.495),
                    xycoords="axes fraction", textcoords="axes fraction",
                    arrowprops=dict(arrowstyle="-|>", color=DE, lw=1.3,
                                    linestyle="dashed"), zorder=4)
    plt.tight_layout()
    save(fig, "figB_methodology_diagram")

# ═════════════════════════════════════════════════════════════════════════════
# FIG C — Workflow Diagram
# ═════════════════════════════════════════════════════════════════════════════
def figC():
    fig, ax = plt.subplots(figsize=(17, 9.5))
    ax.set_xlim(0, 17); ax.set_ylim(0, 9.5); ax.axis("off")
    fig.patch.set_facecolor("#f8f9fa")

    DATA_C = "#1565C0"; PROC_C = "#1E8449"
    EVAL_C = "#C62828"; DIAG_C = "#5D6D7E"

    def wbox(x, y, w, h, title, body, color):
        rect = FancyBboxPatch((x, y), w, h,
                              boxstyle="round,pad=0.12",
                              facecolor=color, edgecolor="white",
                              linewidth=1.6, alpha=0.92, zorder=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h - 0.38, title,
                ha="center", va="top", fontsize=10.5, fontweight="bold",
                color="white", multialignment="center", zorder=3)
        ax.text(x + w/2, y + h/2 - 0.15, body,
                ha="center", va="center", fontsize=8.8, color="white",
                multialignment="center", linespacing=1.42, zorder=3)

    def warr(x1, y, x2, color, lbl=None):
        ax.annotate("", xy=(x2, y), xytext=(x1, y),
                    arrowprops=dict(arrowstyle="-|>", color=color,
                                    lw=2.2, mutation_scale=14), zorder=4)
        if lbl:
            ax.text((x1+x2)/2, y + 0.23, lbl,
                    ha="center", fontsize=8.5, color=color, style="italic")

    # Top row
    top_y = 5.8; box_h = 2.4; bw = 2.8
    configs = [
        (0.35, "Data\nAcquisition",
         "5 datasets\nEdge + CIC (source)\nTON + BoT + N-BaIoT\n(held-out)", DATA_C),
        (3.50, "Feature\nHarmonization",
         "18 NetFlow v2 features\nSynonym-map projection\nRobustScaler (train only)", PROC_C),
        (6.65, "Model\nTraining",
         "XGBoost Cascade\n5-fold OOF meta-learner\n5 random seeds", PROC_C),
        (9.80, "Zero-Shot\nEvaluation",
         "BoT-IoT, N-BaIoT, TON-IoT\nNo target labels used\nMacro F1 + AUROC + σ", EVAL_C),
        (12.95, "Results &\nAnalysis",
         "In-domain F1=0.961\nBoT-IoT F1=0.500\nσ=0.000 (deterministic)", EVAL_C),
    ]
    for xc, title, body, color in configs:
        wbox(xc, top_y, bw, box_h, title, body, color)

    arrow_info = [
        (0.35+bw, 3.50, PROC_C, "harmonize"),
        (3.50+bw, 6.65, PROC_C, "train"),
        (6.65+bw, 9.80, EVAL_C, "evaluate"),
        (9.80+bw, 12.95, EVAL_C, "report"),
    ]
    for x1, x2, c, lbl in arrow_info:
        warr(x1, top_y + box_h/2, x2, c, lbl)

    # Bottom row
    bot_y = 1.0; bot_h = 2.8; bot_bw = 4.60
    diags = [
        (0.35, "LODO Toxicity\nAnalysis",
         "TON-IoT exclusion:\n0.077 → 0.127 (+65.3%)\nAll configs catastrophic"),
        (5.85, "Distribution\nShift Analysis",
         "Wasserstein 99.02 (BoT)\nWasserstein 30.0 (N-BaIoT)\nFeature space divergence"),
        (11.35, "Adaptation\nBaselines",
         "CORAL, DANN, RLLS, DQL\nAll fail: benign_F1=0\nCORAL hurts N-BaIoT"),
    ]
    for xc, title, body in diags:
        wbox(xc, bot_y, bot_bw, bot_h, title, body, DIAG_C)

    for src_x in [8.05, 11.20]:
        ax.annotate("", xy=(src_x, bot_y + bot_h), xytext=(src_x, top_y),
                    arrowprops=dict(arrowstyle="-|>", color=DIAG_C, lw=1.6,
                                    linestyle="dashed", mutation_scale=12), zorder=5)

    ax.set_title("Fig. C  Research Workflow: Cross-Dataset IoT NIDS Benchmark",
                 fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout()
    save(fig, "figC_workflow_diagram")


# ═════════════════════════════════════════════════════════════════════════════
# MAIN
# ═════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    print(f"\nGenerating 13 submission-ready figures → {OUT}\n")
    fig1();  fig2();  fig3();  fig4();  fig5()
    fig6();  fig7();  fig8();  fig9();  fig10()
    figA();  figB();  figC()
    print(f"\nDone — all 13 figures saved to {OUT}\n")


Generating 13 submission-ready figures → figures_submission

  ✓  fig1_cross_domain_gap
  ✓  fig2_class_distribution
  ✓  fig3_lodo_toxicity
  ✓  fig4_confusion_matrices
  ✓  fig5_multiseed_validation
  ✓  fig6_adaptation_methods
  ✓  fig7_auroc_analysis
  ✓  fig8_sample_size_sensitivity
  ✓  fig9_tsne_distribution
  ✓  fig10_class_composition
  ✓  figA_pipeline_diagram


C:\Users\HP\AppData\Local\Temp\ipykernel_7216\3809289739.py:904: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  ✓  figB_methodology_diagram
  ✓  figC_workflow_diagram

Done — all 13 figures saved to figures_submission



In [25]:
"""
Submission-Ready Figure Generator — Cross-Dataset IoT NIDS
All 13 figures at 300 DPI, publication quality.
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

OUT = Path("figures_2")
OUT.mkdir(parents=True, exist_ok=True)

# ── Global rcParams ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          11,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "axes.labelweight":   "bold",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.9,
    "axes.edgecolor":     "#444444",
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "xtick.major.size":   4,
    "ytick.major.size":   4,
    "legend.framealpha":  0.92,
    "legend.edgecolor":   "#cccccc",
    "legend.fontsize":    9,
    "figure.dpi":         300,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.10,
})

# ── Palette ──────────────────────────────────────────────────────────────────
SRC_DARK  = "#1565C0"
SRC_MED   = "#2E86AB"
SRC_LIGHT = "#7BC8F6"
HLD_DARK  = "#C62828"
HLD_MED   = "#C73E1D"
HLD_LIGHT = "#FF8B8B"
BEN_GR    = "#8BC34A"
ARR_BLUE  = "#1A5276"
ARR_GRN   = "#1E8449"
BASE_CLR  = "#555555"

def save(fig, name):
    fig.savefig(OUT / f"{name}.png")
    fig.savefig(OUT / f"{name}.pdf")
    plt.close(fig)
    print(f"  ✓  {name}")

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def fig_bg(fig):
    fig.patch.set_facecolor("#f8f9fa")

# ── Verified data ────────────────────────────────────────────────────────────
INDOMAIN_MF1 = 0.9610;  INDOMAIN_BF1 = 0.9275
BOTIOT_MF1   = 0.4997;  BOTIOT_BF1   = 0.0000;  BOTIOT_AUROC  = 0.0882
NBAIOT_MF1   = 0.4734;  NBAIOT_BF1   = 0.0000;  NBAIOT_AUROC  = 0.5000
TONIOT_MF1   = 0.8808

LODO = {
    "All 3 sources":      0.0768,
    "Excl. TON-IoT":      0.1270,
    "Excl. Edge-IIoTset": 0.0314,
    "Excl. CIC-IoT2023":  0.0147,
}

ADAPT_BOTIOT = {
    "Baseline XGB\n(no adapt.)": 0.4997,
    "CORAL":  0.1112,
    "DANN":   0.4951,
    "RLLS":   0.4984,
    "DQL":    0.4987,
}
ADAPT_NBAIOT = {
    "Baseline XGB\n(no adapt.)": 0.4734,
    "CORAL":  0.0917,
    "DANN":   0.4734,
}

DATASETS = {
    "Edge-IIoTset": {"bp": 15.4,  "n": 157800, "ben": 24301,  "mal": 133499, "role": "source"},
    "CIC-IoT2023":  {"bp":  2.3,  "n": 300000, "ben": 6903,   "mal": 293097, "role": "source"},
    "TON-IoT":      {"bp": 23.7,  "n": 211043, "ben": 50000,  "mal": 161043, "role": "target"},
    "BoT-IoT":      {"bp":  0.028,"n": 285000, "ben": 81,     "mal": 284919, "role": "target"},
    "N-BaIoT":      {"bp": 21.7,  "n": 228183, "ben": 49548,  "mal": 178635, "role": "target"},
}

METHOD_COLORS = {
    "Baseline XGB\n(no adapt.)": "#7f8c8d",
    "CORAL": "#9b59b6",
    "DANN":  "#3498db",
    "RLLS":  "#e67e22",
    "DQL":   "#e74c3c",
}



# ═════════════════════════════════════════════════════════════════════════════
# FIG 10 — Class Composition
# ═════════════════════════════════════════════════════════════════════════════
def fig10():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig_bg(fig)
    fig.suptitle("Fig. 10  Class Composition and Benign Rate Across Datasets\n"
                 "(verified counts from cascade experiment label inventory)",
                 fontsize=12, fontweight="bold")

    dnames   = list(DATASETS.keys())
    roles    = [DATASETS[d]["role"] for d in dnames]
    bpcts    = [DATASETS[d]["bp"]   for d in dnames]
    bens     = [DATASETS[d]["ben"]  for d in dnames]
    mals     = [DATASETS[d]["mal"]  for d in dnames]
    bar_cols = [SRC_DARK if r == "source" else HLD_DARK for r in roles]
    x        = np.arange(5)
    tick_lbl = [f"{d}\n({'source' if r=='source' else 'held-out'})"
                for d, r in zip(dnames, roles)]

    # Panel A
    ax = axes[0]; ax.set_facecolor("#ffffff")
    ax.bar(x, bpcts, color=bar_cols, alpha=0.88, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    ax.set_yscale("log"); ax.set_ylim(0.005, 500)
    for i, (v, c) in enumerate(zip(bpcts, bar_cols)):
        lbl = f"{v:.3f}%" if v < 1 else f"{v:.1f}%"
        ax.text(i, v * 2.6, lbl, ha="center", va="bottom",
                fontsize=8.5, fontweight="bold", color=c)
    ax.set_title("(A)  Benign Rate per Dataset (Log Scale)", fontweight="bold")
    ax.set_ylabel("Benign Rate (%)", labelpad=6)
    ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8.5)
    patches_a = [mpatches.Patch(color=SRC_DARK, label="Source (training)"),
                 mpatches.Patch(color=HLD_DARK, label="Held-out (test)")]
    ax.legend(handles=patches_a, loc="lower right", fontsize=9)
    ax.grid(axis="y", alpha=0.22, which="both", linewidth=0.6, zorder=0)
    despine(ax)

    # Panel B
    ax = axes[1]; ax.set_facecolor("#ffffff")
    mal_cols = [SRC_DARK if r == "source" else HLD_DARK for r in roles]
    ax.bar(x, mals, color=mal_cols, alpha=0.85, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    ax.bar(x, bens, bottom=mals, color=BEN_GR, alpha=0.9, edgecolor="#333",
           linewidth=0.7, width=0.65, zorder=3)
    for i, (b, m, bp) in enumerate(zip(bens, mals, bpcts)):
        lbl = f"{bp:.3f}%" if bp < 1 else f"{bp:.1f}%"
        ax.text(i, m + b + 4500, lbl, ha="center", va="bottom",
                fontsize=8, fontweight="bold", color="#33691E")
    ax.set_title("(B)  Absolute Class Counts\n"
                 "(BoT-IoT benign bar is sub-pixel: only 81 samples)",
                 fontweight="bold")
    ax.set_ylabel("Sample Count", labelpad=6)

    ax.set_xticks(x); ax.set_xticklabels(tick_lbl, fontsize=8.5)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    patches_b = [
        mpatches.Patch(color=SRC_DARK, alpha=0.85, label="Malicious – Source"),
        mpatches.Patch(color=HLD_DARK, alpha=0.85, label="Malicious – Held-out"),
        mpatches.Patch(color=BEN_GR,   alpha=0.90, label="Benign"),
    ]
    ax.legend(handles=patches_b, loc="upper right", fontsize=9, bbox_to_anchor=(1.0, 0.45))
    # ax.legend(loc="lower right", fontsize=9, framealpha=0.95, bbox_to_anchor=(1.0, 0.0))
    ax.grid(axis="y", alpha=0.22, linewidth=0.7, zorder=0)
    despine(ax)

    plt.tight_layout()
    save(fig, "fig10_class_composition")




# ═════════════════════════════════════════════════════════════════════════════
# MAIN
# ═════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    print(f"\nGenerating 13 submission-ready figures → {OUT}\n")
    fig10()
    print(f"\nDone — all 13 figures saved to {OUT}\n")


Generating 13 submission-ready figures → figures_2

  ✓  fig10_class_composition

Done — all 13 figures saved to figures_2



In [26]:
"""
Cross-Dataset IoT NIDS — Complete Figure Generation
All figures: fig1–fig8 + figA + figB + figC
Output: /home/claude/figures/
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
import os

OUT = "/home/claude/figures"
os.makedirs(OUT, exist_ok=True)

# ── Global rcParams ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.size": 11, "font.family": "DejaVu Sans",
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
})

# ── COLOR CONSTANTS ───────────────────────────────────────────────────────────
C_SRC    = "#2E86AB"   # source blue
C_HLD    = "#C73E1D"   # held-out red
C_SRC_L  = "#7BC8F6"   # light source
C_HLD_L  = "#FF8B8B"   # light held-out
C_GRN    = "#8BC34A"   # benign green
C_ARROW  = "#1A5276"   # arrow dark blue
C_LABEL  = "#1E8449"   # label dark green
C_LGRAY  = "#e8e8e8"
C_DGRAY  = "#c2cdd6"
INK      = "#1a1a1a"
SUBINK   = "#555555"


def save(fig, name):
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=150, facecolor="white", bbox_inches="tight", pad_inches=0.15)
    print(f"  ✓ {name}")
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
# FIG A — Pipeline Diagram
# ─────────────────────────────────────────────────────────────────────────────
def figA_pipeline_diagram():
    fig = plt.figure(figsize=(16, 9))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 9)
    ax.axis("off")
    fig.patch.set_facecolor("white")

    # Colors
    C_source = "#1565C0"
    C_feat   = "#2E7D32"
    C_model  = "#6A1B9A"
    C_target = "#B71C1C"
    C_adapt  = "#5D4037"
    C_lodo   = "#37474F"

    def rbox(x, y, w, h, fc, ec="#222222", lw=1.4, radius=0.18, ls="-"):
        ax.add_patch(FancyBboxPatch((x, y), w, h,
            boxstyle=f"round,pad=0,rounding_size={radius}",
            fc=fc, ec=ec, lw=lw, linestyle=ls, zorder=2))

    def ctext(x, y, w, h, lines, sizes, colors=None, weights=None):
        n = len(lines)
        for i, (line, fs) in enumerate(zip(lines, sizes)):
            fy = y + h - (i + 0.5) * h / n
            col = colors[i] if colors else "white"
            fw  = weights[i] if weights else "normal"
            ax.text(x + w/2, fy, line, ha="center", va="center",
                    fontsize=fs, color=col, fontweight=fw, zorder=3,
                    fontfamily="DejaVu Sans")

    def arrow(x0, y0, x1, y1, color, lw=1.8, ls="-", label="", label_c="gray"):
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                   linestyle=ls, mutation_scale=12), zorder=5)
        if label:
            mx, my = (x0+x1)/2, (y0+y1)/2
            ax.text(mx + 0.12, my, label, ha="left", va="center",
                    fontsize=8, color=label_c, style="italic", zorder=5)

    # ── SOURCE DATASETS (left column) ────────────────────────────────────────
    rbox(0.15, 5.60, 2.70, 1.55, C_source)
    ctext(0.15, 5.60, 2.70, 1.55,
          ["Edge-IIoTset", "157,800 samples", "benign 15.4%"],
          [11.5, 10, 9.5], weights=["bold","normal","normal"])

    rbox(0.15, 3.90, 2.70, 1.55, C_source)
    ctext(0.15, 3.90, 2.70, 1.55,
          ["CIC-IoT2023", "300,000 samples", "benign 2.3%"],
          [11.5, 10, 9.5], weights=["bold","normal","normal"])

    ax.text(1.50, 3.70, "SOURCE (train)  —  457,800 total",
            ha="center", va="top", fontsize=9, color=C_source,
            fontweight="bold", zorder=3)

    # ── FEATURE HARMONIZATION ────────────────────────────────────────────────
    rbox(3.20, 3.55, 3.40, 3.00, C_feat)
    ctext(3.20, 3.55, 3.40, 3.00,
          ["Feature Harmonization", "18 NetFlow v2 features", "RobustScaler",
           "(fit on train only)"],
          [12.5, 10.5, 10.5, 9.5], weights=["bold","normal","normal","italic"])

    # ── MODEL ────────────────────────────────────────────────────────────────
    rbox(7.00, 3.40, 3.60, 3.20, C_model)
    ctext(7.00, 3.40, 3.60, 3.20,
          ["XGBoost Cascade", "+ 5-fold OOF", "Meta-Learner", "",
           "seeds: 42/123/456/789/999"],
          [13, 11, 11, 8, 9.5], weights=["bold","normal","normal","normal","normal"])

    # ── TARGET BOXES (right column) ──────────────────────────────────────────
    rbox(11.00, 6.10, 4.70, 1.95, C_target)
    ctext(11.00, 6.10, 4.70, 1.95,
          ["BoT-IoT  (held-out)", "Macro F1=0.4997  |  AUROC=0.0882",
           "benign_F1=0.000  |  sigma=0.0", "Wasserstein dist=99.02"],
          [11.5, 9.5, 9.5, 9.5], weights=["bold","normal","normal","normal"])

    rbox(11.00, 3.90, 4.70, 1.95, C_target)
    ctext(11.00, 3.90, 4.70, 1.95,
          ["N-BaIoT  (held-out)", "Macro F1=0.4734  |  AUROC=0.500",
           "benign_F1=0.000  |  sigma=0.0", "Benign support: 49k / 228k (21.7%)"],
          [11.5, 9.5, 9.5, 9.5], weights=["bold","normal","normal","normal"])

    rbox(11.00, 1.60, 4.70, 1.95, C_target)
    ctext(11.00, 1.60, 4.70, 1.95,
          ["TON-IoT  (held-out)", "211,043 samples  |  benign 23.7%",
           "LODO excl. => F1: 0.077->0.127",
           "(negative transfer source)"],
          [11.5, 9.5, 9.5, 9.5], weights=["bold","normal","normal","italic"])

    # ── DOMAIN ADAPTATION ────────────────────────────────────────────────────
    rbox(7.00, 0.35, 3.60, 2.60, C_adapt)
    ctext(7.00, 0.35, 3.60, 2.60,
          ["Domain Adaptation",
           "CORAL:  F1=0.111 (BoT-IoT)",
           "DANN:   F1=0.495 (BoT-IoT)",
           "RLLS:   F1=0.498  |  DQL: F1=0.499",
           "All fail: benign_F1=0"],
          [11.5, 9, 9, 9, 9], weights=["bold","normal","normal","normal","bold"])

    # ── LODO BOX ────────────────────────────────────────────────────────────
    rbox(0.15, 0.35, 6.60, 1.80, C_lodo)
    ctext(0.15, 0.35, 6.60, 1.80,
          ["LODO Protocol: Leave-One-Dataset-Out",
           "All 3 sources: BoT-IoT F1=0.077     |     Excl. TON-IoT: F1=0.127  (+65.3% relative)",
           "Excl. Edge-IIoTset: F1=0.031     |     Excl. CIC-IoT2023: F1=0.015"],
          [11.5, 9, 9], weights=["bold","normal","normal"])

    # ── ARROWS ───────────────────────────────────────────────────────────────
    # Sources → Feature Harm
    arrow(2.85, 6.38, 3.20, 5.05, C_source)
    arrow(2.85, 4.68, 3.20, 5.05, C_source)
    # Feature → Model
    arrow(6.60, 5.05, 7.00, 5.00, C_feat, label="train\npool", label_c=C_feat)
    # Model → Targets
    arrow(10.60, 5.50, 11.00, 7.08, C_target, label="zero-shot", label_c=C_target)
    arrow(10.60, 5.00, 11.00, 4.88, C_target, label="zero-shot", label_c=C_target)
    arrow(10.60, 4.50, 11.00, 2.58, C_target, label="zero-shot", label_c=C_target)
    # Model → Adapt (post-hoc)
    arrow(8.80, 3.40, 8.80, 2.95, C_adapt, label="post-hoc", label_c=C_adapt)
    # LODO dashed
    ax.annotate("", xy=(4.45, 2.15), xytext=(4.45, 3.55),
                arrowprops=dict(arrowstyle="-|>", color=C_lodo, lw=1.4,
                                linestyle="--", mutation_scale=10), zorder=5)
    ax.text(4.60, 2.90, "excl. one\nsource", ha="left", va="center",
            fontsize=8, color=C_lodo, style="italic", zorder=5)

    # ── TITLE ────────────────────────────────────────────────────────────────
    ax.text(8.00, 8.80,
            "Pipeline: Cross-Dataset IoT Intrusion Detection",
            ha="center", va="center", fontsize=13, fontweight="bold", color=INK, zorder=3)
    ax.text(8.00, 8.45,
            "Source: Edge-IIoTset (157,800) + CIC-IoT2023 (300,000) = 457,800 total  |  "
            "Held-Out: TON-IoT, BoT-IoT, N-BaIoT (zero-shot)",
            ha="center", va="center", fontsize=10, color=INK, zorder=3)

    # ── LEGEND ───────────────────────────────────────────────────────────────
    leg_items = [
        (C_source, "Source Datasets"), (C_feat, "Feature Processing"),
        (C_model, "Model (XGBoost Cascade)"), (C_target, "Held-Out Evaluation"),
        (C_adapt, "Adaptation Baselines"), (C_lodo, "LODO Protocol"),
    ]
    for i, (c, lbl) in enumerate(leg_items):
        col, row = i % 2, i // 2
        lx = 10.70 + col * 2.55
        ly = 0.42 + (2 - row) * 0.36
        ax.add_patch(FancyBboxPatch((lx, ly - 0.13), 0.32, 0.26,
                     boxstyle="round,pad=0,rounding_size=0.04",
                     fc=c, ec="none", zorder=3))
        ax.text(lx + 0.40, ly, lbl, ha="left", va="center",
                fontsize=8.5, color=INK, zorder=3)

    ax.set_ylim(0.10, 9.10)
    fig.savefig(os.path.join(OUT, "figA_pipeline_diagram.png"),
                dpi=150, facecolor="white", bbox_inches="tight", pad_inches=0.12)
    print("  ✓ figA_pipeline_diagram.png")
    plt.close(fig)



# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("Generating all figures...")
   
    figA_pipeline_diagram()

    print(f"\nAll figures saved to: {OUT}")

Generating all figures...


ValueError: weight='italic' is invalid